# Visual-language temporal boundary alignment
This notebook is self-bootstrapping. Because the source repository is private, add a Colab secret named `GITHUB_TOKEN` with read access to `sanay-tralshawala/video-language-temporal-boundaries` (key icon in the left sidebar), or enter a token when prompted. The token is used only for cloning and is not saved in Git configuration.

In [ ]:
# Clone the private source repository, enter it, and install dependencies.
from pathlib import Path
import base64
import getpass
import os
import subprocess
import sys

REPO = 'sanay-tralshawala/video-language-temporal-boundaries'
REPO_DIR = Path('/content/video-language-temporal-boundaries')

if not (REPO_DIR / 'requirements-colab.txt').exists():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not token:
        token = getpass.getpass('GitHub token with read access to the private repository: ')
    if not token:
        raise RuntimeError('A GitHub token is required because this repository is private.')
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    command = ['git', '-c', f'http.extraHeader=AUTHORIZATION: basic {credential}', 'clone', '--depth', '1', f'https://github.com/{REPO}.git', str(REPO_DIR)]
    completed = subprocess.run(command)
    token = credential = None
    if completed.returncode != 0:
        raise RuntimeError('Clone failed. Check that GITHUB_TOKEN can read the private repository.')

os.chdir(REPO_DIR)
requirements = REPO_DIR / 'requirements-colab.txt'
pipeline_file = REPO_DIR / 'video_boundary_pipeline.py'
if not requirements.is_file() or not pipeline_file.is_file():
    raise FileNotFoundError(f'Incomplete checkout at {REPO_DIR}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Repository ready:', REPO_DIR)
print('Requirements file:', requirements)

In [ ]:
# Mount persistent storage and report the accelerator.
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Dataset layout
Put each video beside a same-stem `.srt` or `.vtt` file in Drive, for example `B6fvT2LKEDI.mp4` and `B6fvT2LKEDI.srt`. The validation cell below stops before loading models if no valid pairs are found.

In [ ]:
# Edit this configuration block.
from pathlib import Path
from video_boundary_pipeline import discover_videos

DATASET_DIR = '/content/drive/MyDrive/Video-MME-long-subset'
OUTPUT_DIR = '/content/drive/MyDrive/video_boundary_results'
VIDEO_LIST = None       # or ['B6fvT2LKEDI.mp4', ...]
METADATA_CSV = None     # optional: video_path, subtitle_path[, video_id]
FRAME_INTERVAL = 2.0
SUBTITLE_WINDOW = 15.0

if not Path(DATASET_DIR).is_dir():
    raise FileNotFoundError(f'DATASET_DIR does not exist: {DATASET_DIR}')
pairs = discover_videos(DATASET_DIR, VIDEO_LIST, METADATA_CSV, max_videos=30)
if not pairs:
    raise RuntimeError('No same-stem video/subtitle pairs found. See the dataset-layout cell above.')
print(f'Validated {len(pairs)} video/subtitle pairs')
for pair in pairs[:5]:
    print(' ', pair.video_path.name, '+', pair.subtitle_path.name)

In [ ]:
# Run the complete cached analysis.
from video_boundary_pipeline import PipelineConfig, run_pipeline

config = PipelineConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_DIR,
    frame_interval=FRAME_INTERVAL, subtitle_window=SUBTITLE_WINDOW,
    max_videos=30,
)
results = run_pipeline(config, video_list=VIDEO_LIST, metadata_csv=METADATA_CSV)
display(results['alignment'])
display(results['stats'])